<a href="https://colab.research.google.com/github/OlaYounisACCA/SDS/blob/Dev/L7-SDS/Lab%203/Notebooks/Database_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from datetime import timedelta
import os

# ------------------------------------------------------------
# NovaCore Solutions Ltd - Lab 3 SQL Environment
# Creates: novacore_lab3_database.duckdb
# Tables:
#   customers     2,000
#   products        150
#   invoices     18,000
#   payments     25,000
#   transactions 35,000
# ------------------------------------------------------------

np.random.seed(3307)

DB_FILE = "novacore_lab3_database.duckdb"
AS_OF_DATE = pd.Timestamp("2026-01-31")

N_CUSTOMERS = 2000
N_PRODUCTS = 150
N_INVOICES = 18000
N_PAYMENTS = 25000
N_TRANSACTIONS = 35000

# Ensure any existing connection is closed before attempting to remove the file
# This handles cases where a previous run might have failed before con.close()
if 'con' in globals() and isinstance(con, duckdb.DuckDBPyConnection):
    try:
        con.close()
        print("Closed previous DuckDB connection to ensure clean file removal.")
    except Exception as e:
        print(f"Could not close previous DuckDB connection: {e}")

if os.path.exists(DB_FILE):
    try:
        os.remove(DB_FILE)
        print(f"Removed existing database file: {DB_FILE}")
    except OSError as e:
        print(f"Error removing {DB_FILE}: {e}. It might be in use.")
        # Re-raise or handle this critical error if removal is essential
        raise

# ------------------------------------------------------------
# Reference data
# ------------------------------------------------------------

industries = [
    "Retail",
    "Manufacturing",
    "Healthcare",
    "Logistics",
    "Financial Services",
    "Education",
    "Energy",
    "Professional Services",
    "Technology",
    "Public Services"
]

regions = [
    "North Region",
    "South Region",
    "East Region",
    "West Region",
    "Central Region",
    "Global Accounts"
]

segments = ["Enterprise", "Mid-Market", "Small Business"]
credit_ratings = ["A", "B", "C", "D"]

payment_methods = [
    "Bank Transfer",
    "Card",
    "Digital Payment",
    "Direct Debit"
]

product_categories = [
    "Software",
    "Consulting",
    "Support",
    "Hardware",
    "Managed Services",
    "Data Services",
    "Analytics Services",
    "Training Services"
]

industry_multipliers = {
    "Retail": 0.95,
    "Manufacturing": 1.20,
    "Healthcare": 1.05,
    "Logistics": 1.00,
    "Financial Services": 1.35,
    "Education": 0.78,
    "Energy": 1.45,
    "Professional Services": 1.12,
    "Technology": 1.30,
    "Public Services": 0.88
}

region_multipliers = {
    "North Region": 1.03,
    "South Region": 0.92,
    "East Region": 0.98,
    "West Region": 1.08,
    "Central Region": 0.95,
    "Global Accounts": 1.42
}

segment_multipliers = {
    "Enterprise": 2.15,
    "Mid-Market": 1.00,
    "Small Business": 0.45
}

segment_credit_limits = {
    "Enterprise": (150000, 800000),
    "Mid-Market": (40000, 220000),
    "Small Business": (10000, 70000)
}

credit_rating_prob_by_segment = {
    "Enterprise": [0.38, 0.40, 0.17, 0.05],
    "Mid-Market": [0.22, 0.45, 0.25, 0.08],
    "Small Business": [0.14, 0.35, 0.35, 0.16]
}

name_prefixes = [
    "Aster", "BluePeak", "Cobalt", "Delta", "Eclipse", "Futura", "Granite",
    "Helio", "Ion", "Juniper", "Keystone", "Lumina", "Meridian", "Nimbus",
    "Orchid", "Pinnacle", "Quantum", "Redwood", "Summit", "Terra", "Umbra",
    "Vector", "Willow", "Zenith", "Aurora", "Crescent", "Harbour", "Silverline"
]

name_suffixes = [
    "Group", "Holdings", "Systems", "Partners", "Enterprises", "Networks",
    "Operations", "Services", "Industries", "Solutions", "Alliance", "Works",
    "Ventures", "Collective"
]

product_prefixes = [
    "Nova", "Core", "Axis", "Prime", "Pulse", "Vector", "Nexus", "Optima",
    "Helix", "Quantum", "Summit", "Orion"
]

product_suffixes = [
    "Platform", "Suite", "Licence", "Service", "Module", "Gateway",
    "Desk", "Hub", "Engine", "Toolkit"
]

# ------------------------------------------------------------
# Customers
# ------------------------------------------------------------

customers = []

for i in range(1, N_CUSTOMERS + 1):
    segment = np.random.choice(segments, p=[0.20, 0.45, 0.35])
    industry = np.random.choice(
        industries,
        p=[0.11, 0.13, 0.10, 0.09, 0.11, 0.08, 0.08, 0.11, 0.12, 0.07]
    )
    region = np.random.choice(
        regions,
        p=[0.18, 0.16, 0.17, 0.18, 0.18, 0.13]
    )
    rating = np.random.choice(
        credit_ratings,
        p=credit_rating_prob_by_segment[segment]
    )
    credit_min, credit_max = segment_credit_limits[segment]
    credit_limit = round(np.random.uniform(credit_min, credit_max) / 100) * 100

    since_date = pd.Timestamp("2017-01-01") + pd.to_timedelta(
        int(np.random.randint(0, 2600)),
        unit="D"
    )

    customer_name = (
        f"{np.random.choice(name_prefixes)} "
        f"{np.random.choice(name_suffixes)} "
        f"{np.random.choice(['Ltd', 'Group', 'Company', 'Collective'])}"
    )

    customers.append({
        "customer_id": f"CUST{i:05d}",
        "customer_name": customer_name,
        "industry": industry,
        "customer_segment": segment,
        "region": region,
        "customer_since_date": since_date.date(),
        "credit_rating": rating,
        "credit_limit_CUR": float(credit_limit),
        "customer_value_factor": np.random.lognormal(mean=0.0, sigma=0.35)
    })

customers_df = pd.DataFrame(customers)

# ------------------------------------------------------------
# Products
# ------------------------------------------------------------

products = []

for i in range(1, N_PRODUCTS + 1):
    category = np.random.choice(
        product_categories,
        p=[0.23, 0.16, 0.15, 0.12, 0.14, 0.09, 0.08, 0.03]
    )

    if category == "Software":
        cost = np.random.uniform(40, 180)
        margin = np.random.uniform(1.8, 3.2)
    elif category == "Consulting":
        cost = np.random.uniform(180, 650)
        margin = np.random.uniform(1.7, 2.7)
    elif category == "Support":
        cost = np.random.uniform(30, 160)
        margin = np.random.uniform(1.6, 2.5)
    elif category == "Hardware":
        cost = np.random.uniform(250, 1200)
        margin = np.random.uniform(1.25, 1.75)
    elif category == "Managed Services":
        cost = np.random.uniform(150, 500)
        margin = np.random.uniform(1.7, 2.8)
    elif category == "Data Services":
        cost = np.random.uniform(90, 350)
        margin = np.random.uniform(1.8, 3.0)
    elif category == "Analytics Services":
        cost = np.random.uniform(160, 520)
        margin = np.random.uniform(1.9, 3.3)
    else:
        cost = np.random.uniform(60, 220)
        margin = np.random.uniform(1.4, 2.2)

    selling_price = round((cost * margin) / 10) * 10
    unit_cost = round(cost / 10) * 10

    product_name = (
        f"{np.random.choice(product_prefixes)} "
        f"{category} "
        f"{np.random.choice(product_suffixes)}"
    )

    products.append({
        "product_id": f"PROD{i:04d}",
        "product_category": category,
        "product_name": product_name,
        "unit_cost_CUR": float(unit_cost),
        "selling_price_CUR": float(selling_price)
    })

products_df = pd.DataFrame(products)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def choose_payment_terms(segment, rating):
    if segment == "Enterprise":
        terms = np.random.choice([30, 45, 60], p=[0.35, 0.45, 0.20])
    elif segment == "Mid-Market":
        terms = np.random.choice([15, 30, 45], p=[0.15, 0.65, 0.20])
    else:
        terms = np.random.choice([15, 30], p=[0.35, 0.65])

    if rating == "D" and np.random.random() < 0.20:
        terms = max(15, terms - 15)

    return int(terms)


def payment_delay_by_rating(rating, segment):
    if rating == "A":
        delay = np.random.normal(loc=-4, scale=7)
    elif rating == "B":
        delay = np.random.normal(loc=3, scale=10)
    elif rating == "C":
        delay = np.random.normal(loc=17, scale=18)
    else:
        delay = np.random.normal(loc=42, scale=32)

    if segment == "Enterprise":
        delay -= np.random.normal(loc=2, scale=3)
    elif segment == "Small Business":
        delay += np.random.normal(loc=5, scale=5)

    return int(np.clip(round(delay), -25, 180))


# ------------------------------------------------------------
# Invoices
# ------------------------------------------------------------

records = []
start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2025-12-31")
date_span = (end_date - start_date).days

customer_weights = customers_df["customer_segment"].map({
    "Enterprise": 2.6,
    "Mid-Market": 1.2,
    "Small Business": 0.65
}).values

customer_weights = customer_weights / customer_weights.sum()

for i in range(1, N_INVOICES + 1):
    customer = customers_df.sample(
        1,
        weights=customer_weights
    ).iloc[0]

    product = products_df.sample(1).iloc[0]

    invoice_date = start_date + pd.to_timedelta(
        int(np.random.randint(0, date_span + 1)),
        unit="D"
    )

    terms = choose_payment_terms(
        customer["customer_segment"],
        customer["credit_rating"]
    )

    due_date = invoice_date + pd.to_timedelta(terms, unit="D")

    if customer["customer_segment"] == "Enterprise":
        quantity = int(np.clip(np.random.lognormal(mean=3.15, sigma=0.75), 4, 320))
    elif customer["customer_segment"] == "Mid-Market":
        quantity = int(np.clip(np.random.lognormal(mean=2.35, sigma=0.65), 2, 140))
    else:
        quantity = int(np.clip(np.random.lognormal(mean=1.65, sigma=0.55), 1, 55))

    standard_amount = quantity * product["selling_price_CUR"]

    multiplier = (
        industry_multipliers[customer["industry"]]
        * region_multipliers[customer["region"]]
        * segment_multipliers[customer["customer_segment"]]
        * customer["customer_value_factor"]
    )

    commercial_adjustment = np.clip(np.random.normal(1.0, 0.07), 0.75, 1.25)
    amount = standard_amount * multiplier * commercial_adjustment

    # Genuine high-value enterprise invoices
    if (
        customer["customer_segment"] == "Enterprise"
        and customer["region"] == "Global Accounts"
        and np.random.random() < 0.045
    ):
        amount *= np.random.uniform(2.0, 4.8)

    # Small recurring invoices
    if np.random.random() < 0.045:
        amount *= np.random.uniform(0.30, 0.60)

    amount = float(np.clip(amount, 150, 950000))
    amount = round(amount / 10) * 10

    records.append({
        "invoice_id": f"INV{i:08d}",
        "customer_id": customer["customer_id"],
        "product_id": product["product_id"],
        "invoice_date": invoice_date.date(),
        "due_date": due_date.date(),
        "quantity": quantity,
        "invoice_amount_CUR": float(amount),
        "payment_status": "Pending"
    })

invoices_df = pd.DataFrame(records)

# ------------------------------------------------------------
# Assign payment outcomes and exact payment counts
# ------------------------------------------------------------

status_choices = [
    "Paid",
    "Partially Paid",
    "Unpaid",
    "Overdue",
    "Written Off",
    "Overpaid"
]

status_probs = [0.55, 0.15, 0.07, 0.17, 0.02, 0.04]

invoices_df["payment_status"] = np.random.choice(
    status_choices,
    size=N_INVOICES,
    p=status_probs
)

# Initial number of payments per invoice
payment_counts = []

for status in invoices_df["payment_status"]:
    if status == "Paid":
        payment_counts.append(np.random.choice([1, 2], p=[0.74, 0.26]))
    elif status == "Partially Paid":
        payment_counts.append(np.random.choice([1, 2, 3], p=[0.25, 0.50, 0.25]))
    elif status == "Unpaid":
        payment_counts.append(0)
    elif status == "Overdue":
        payment_counts.append(np.random.choice([0, 1, 2], p=[0.40, 0.45, 0.15]))
    elif status == "Written Off":
        payment_counts.append(np.random.choice([0, 1], p=[0.75, 0.25]))
    elif status == "Overpaid":
        payment_counts.append(np.random.choice([1, 2], p=[0.60, 0.40]))

payment_counts = np.array(payment_counts)

# Adjust to exactly 25,000 payment records
current_count = int(payment_counts.sum())

while current_count < N_PAYMENTS:
    eligible = np.where(
        (invoices_df["payment_status"].isin(["Paid", "Partially Paid", "Overdue", "Overpaid"]))
        & (payment_counts < 4)
    )[0]
    idx = np.random.choice(eligible)
    payment_counts[idx] += 1
    current_count += 1

while current_count > N_PAYMENTS:
    eligible = np.where(payment_counts > 1)[0]
    idx = np.random.choice(eligible)
    payment_counts[idx] -= 1
    current_count -= 1

invoices_df["payment_count"] = payment_counts

# ------------------------------------------------------------
# Payments
# ------------------------------------------------------------

payments = []
payment_id_counter = 1

invoice_lookup = invoices_df.merge(
    customers_df[["customer_id", "customer_segment", "credit_rating"]],
    on="customer_id",
    how="left"
)

for _, inv in invoice_lookup.iterrows():
    n_pay = int(inv["payment_count"])

    if n_pay == 0:
        continue

    amount = float(inv["invoice_amount_CUR"])
    status = inv["payment_status"]

    if status == "Paid":
        total_paid = amount
    elif status == "Partially Paid":
        total_paid = amount * np.random.uniform(0.25, 0.85)
    elif status == "Overdue":
        total_paid = amount * np.random.uniform(0.10, 0.95)
    elif status == "Written Off":
        total_paid = amount * np.random.uniform(0.05, 0.35)
    elif status == "Overpaid":
        total_paid = amount * np.random.uniform(1.01, 1.10)
    else:
        total_paid = 0

    total_paid = round(total_paid / 10) * 10

    weights = np.random.dirichlet(np.ones(n_pay))
    split_amounts = np.round((total_paid * weights) / 10) * 10

    # Adjust rounding difference into final payment
    rounding_diff = total_paid - split_amounts.sum()
    split_amounts[-1] += rounding_diff

    delay = payment_delay_by_rating(
        inv["credit_rating"],
        inv["customer_segment"]
    )

    # Overdue items are usually late
    if status in ["Overdue", "Partially Paid", "Written Off"]:
        delay = max(delay, np.random.randint(5, 100))

    first_payment_date = pd.Timestamp(inv["due_date"]) + pd.to_timedelta(delay, unit="D")

    for j in range(n_pay):
        payment_date = first_payment_date + pd.to_timedelta(
            int(np.random.randint(0, 35 * max(1, n_pay))),
            unit="D"
        )

        # Prevent unrealistically far future dates
        payment_date = min(payment_date, AS_OF_DATE)

        payments.append({
            "payment_id": f"PAY{payment_id_counter:08d}",
            "invoice_id": inv["invoice_id"],
            "payment_date": payment_date.date(),
            "payment_amount_CUR": float(max(split_amounts[j], 0)),
            "payment_method": np.random.choice(
                payment_methods,
                p=[0.58, 0.12, 0.18, 0.12]
            )
        })

        payment_id_counter += 1

payments_df = pd.DataFrame(payments)

assert len(payments_df) == N_PAYMENTS

# ------------------------------------------------------------
# Transactions
# ------------------------------------------------------------

transactions = []
txn_id = 1

# 1. Invoice postings: 18,000
for _, inv in invoices_df.iterrows():
    transactions.append({
        "transaction_id": f"TXN{txn_id:09d}",
        "transaction_date": inv["invoice_date"],
        "customer_id": inv["customer_id"],
        "invoice_id": inv["invoice_id"],
        "transaction_type": "Invoice",
        "amount_CUR": float(inv["invoice_amount_CUR"])
    })
    txn_id += 1

# 2. Payment summary transactions: one per invoice with payments
payment_summary = (
    payments_df.groupby("invoice_id", as_index=False)
    .agg(
        transaction_date=("payment_date", "max"),
        amount_CUR=("payment_amount_CUR", "sum")
    )
)

payment_summary = payment_summary.merge(
    invoices_df[["invoice_id", "customer_id"]],
    on="invoice_id",
    how="left"
)

for _, pay in payment_summary.iterrows():
    transactions.append({
        "transaction_id": f"TXN{txn_id:09d}",
        "transaction_date": pay["transaction_date"],
        "customer_id": pay["customer_id"],
        "invoice_id": pay["invoice_id"],
        "transaction_type": "Payment Summary",
        "amount_CUR": float(-pay["amount_CUR"])
    })
    txn_id += 1

# 3. Adjustments, credit notes and write-offs to reach exactly 35,000
remaining = N_TRANSACTIONS - len(transactions)

adjustment_candidates = invoices_df.sample(
    n=remaining,
    replace=True,
    random_state=3310
).reset_index(drop=True)

for _, inv in adjustment_candidates.iterrows():
    r = np.random.random()

    if inv["payment_status"] == "Written Off" or r < 0.25:
        txn_type = "Write-off"
        amount = -float(inv["invoice_amount_CUR"]) * np.random.uniform(0.05, 0.40)
        txn_date = pd.Timestamp(inv["due_date"]) + pd.to_timedelta(
            int(np.random.randint(45, 180)),
            unit="D"
        )
    elif r < 0.60:
        txn_type = "Credit Note"
        amount = -float(inv["invoice_amount_CUR"]) * np.random.uniform(0.02, 0.15)
        txn_date = pd.Timestamp(inv["invoice_date"]) + pd.to_timedelta(
            int(np.random.randint(5, 90)),
            unit="D"
        )
    else:
        txn_type = "Adjustment"
        amount = float(inv["invoice_amount_CUR"]) * np.random.uniform(-0.05, 0.05)
        txn_date = pd.Timestamp(inv["invoice_date"]) + pd.to_timedelta(
            int(np.random.randint(1, 120)),
            unit="D"
        )

    txn_date = min(txn_date, AS_OF_DATE)

    transactions.append({
        "transaction_id": f"TXN{txn_id:09d}",
        "transaction_date": txn_date.date(),
        "customer_id": inv["customer_id"],
        "invoice_id": inv["invoice_id"],
        "transaction_type": txn_type,
        "amount_CUR": float(round(amount / 10) * 10)
    })
    txn_id += 1

transactions_df = pd.DataFrame(transactions)

assert len(transactions_df) == N_TRANSACTIONS

# Remove helper column before database load
invoices_df = invoices_df.drop(columns=["payment_count"])
customers_df = customers_df.drop(columns=["customer_value_factor"])

# ------------------------------------------------------------
# Create DuckDB database
# ------------------------------------------------------------

con = duckdb.connect(DB_FILE)

con.execute("PRAGMA enable_object_cache;")

con.execute("""
CREATE TABLE customers (
    customer_id VARCHAR PRIMARY KEY,
    customer_name VARCHAR,
    industry VARCHAR,
    customer_segment VARCHAR,
    region VARCHAR,
    customer_since_date DATE,
    credit_rating VARCHAR,
    credit_limit_CUR DECIMAL(18,2)
);
""")

con.execute("""
CREATE TABLE products (
    product_id VARCHAR PRIMARY KEY,
    product_category VARCHAR,
    product_name VARCHAR,
    unit_cost_CUR DECIMAL(18,2),
    selling_price_CUR DECIMAL(18,2)
);
""")

con.execute("""
CREATE TABLE invoices (
    invoice_id VARCHAR PRIMARY KEY,
    customer_id VARCHAR,
    product_id VARCHAR,
    invoice_date DATE,
    due_date DATE,
    quantity INTEGER,
    invoice_amount_CUR DECIMAL(18,2),
    payment_status VARCHAR,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

con.execute("""
CREATE TABLE payments (
    payment_id VARCHAR PRIMARY KEY,
    invoice_id VARCHAR,
    payment_date DATE,
    payment_amount_CUR DECIMAL(18,2),
    payment_method VARCHAR,
    FOREIGN KEY (invoice_id) REFERENCES invoices(invoice_id)
);
""")

con.execute("""
CREATE TABLE transactions (
    transaction_id VARCHAR PRIMARY KEY,
    transaction_date DATE,
    customer_id VARCHAR,
    invoice_id VARCHAR,
    transaction_type VARCHAR,
    amount_CUR DECIMAL(18,2),
    FOREIGN KEY (invoice_id) REFERENCES invoices(invoice_id)
);
""")
# ------------------------------------------------------------
# Introduce Data Quality Issues for SQL Cleaning Practice
# ------------------------------------------------------------

# 1. Missing values
missing_idx = customers_df.sample(
    n=20,
    random_state=3307
).index

customers_df.loc[missing_idx, "credit_rating"] = None
customers_df.loc[customers_df.sample(30, random_state=3311).index, "region"] = None
products_df.loc[products_df.sample(10, random_state=3312).index, "product_category"] = None
payments_df.loc[payments_df.sample(50, random_state=3313).index, "payment_method"] = None
transactions_df.loc[transactions_df.sample(25, random_state=3314).index, "transaction_type"] = None

# 2. Text standardisation issues
text_error_idx = customers_df.sample(
    n=30,
    random_state=3308
).index

customers_df.loc[text_error_idx, "customer_segment"] = (
    customers_df.loc[text_error_idx, "customer_segment"]
    .str.lower()
    .str.strip()
)


# 3. One unusually large invoice (outlier)
invoice_outlier_idx = invoices_df.sample(
    n=1,
    random_state=3310
).index

invoices_df.loc[
    invoice_outlier_idx,
    "invoice_amount_CUR"
] = 9999999


# Load the customer master with a unique customer_id.
con.register("customers_df", customers_df)
con.register("products_df", products_df)
con.register("invoices_df", invoices_df)
con.register("payments_df", payments_df)
con.register("transactions_df", transactions_df)

con.execute("INSERT INTO customers SELECT * FROM customers_df;")
con.execute("INSERT INTO products SELECT * FROM products_df;")
con.execute("INSERT INTO invoices SELECT * FROM invoices_df;")
con.execute("INSERT INTO payments SELECT * FROM payments_df;")
con.execute("INSERT INTO transactions SELECT * FROM transactions_df;")

# Helpful views for students
con.execute("""
CREATE VIEW invoice_payment_summary AS
SELECT
    i.invoice_id,
    i.customer_id,
    i.product_id,
    i.invoice_date,
    i.due_date,
    i.quantity,
    i.invoice_amount_CUR,
    i.payment_status,
    COALESCE(SUM(p.payment_amount_CUR), 0) AS total_paid_CUR,
    MAX(p.payment_date) AS latest_payment_date,
    i.invoice_amount_CUR - COALESCE(SUM(p.payment_amount_CUR), 0) AS outstanding_amount_CUR,
    CASE
        WHEN MAX(p.payment_date) IS NULL THEN NULL
        ELSE DATE_DIFF('day', i.due_date, MAX(p.payment_date))
    END AS payment_delay_days
FROM invoices i
LEFT JOIN payments p
    ON i.invoice_id = p.invoice_id
GROUP BY
    i.invoice_id,
    i.customer_id,
    i.product_id,
    i.invoice_date,
    i.due_date,
    i.quantity,
    i.invoice_amount_CUR,
    i.payment_status;
""")

con.execute("""
CREATE VIEW invoice_profitability AS
SELECT
    i.invoice_id,
    i.customer_id,
    i.product_id,
    i.invoice_date,
    i.invoice_amount_CUR,
    i.quantity,
    p.product_category,
    p.product_name,
    p.unit_cost_CUR,
    p.selling_price_CUR,
    i.quantity * p.unit_cost_CUR AS estimated_cost_CUR,
    i.invoice_amount_CUR - (i.quantity * p.unit_cost_CUR) AS gross_profit_CUR,
    CASE
        WHEN i.invoice_amount_CUR = 0 THEN NULL
        ELSE (i.invoice_amount_CUR - (i.quantity * p.unit_cost_CUR)) / i.invoice_amount_CUR
    END AS gross_margin_rate
FROM invoices i
JOIN products p
    ON i.product_id = p.product_id;
""")


print(f"Created DuckDB database: {DB_FILE}")
print("Row counts:")
print("customers:", len(customers_df))
print("products:", len(products_df))
print("invoices:", len(invoices_df))
print("payments:", len(payments_df))
print("transactions:", len(transactions_df))

In [ ]:

con.execute("""
SHOW TABLES;
""").df()

In [ ]:
import duckdb

# Re-establish the connection to the DuckDB database
con = duckdb.connect('novacore_lab3_database.duckdb')

print("\n--- Data Quality Checks ---\n")

# 1. Check for missing values in 'credit_rating'
print("Missing 'credit_rating' values in customers table:")
missing_credit_rating = con.execute("SELECT customer_id, customer_name FROM customers WHERE credit_rating IS NULL;").fetchdf()
print(missing_credit_rating)

print("\nText standardization issues in 'customer_segment' in customers table:")
# 2. Check for text standardization issues in 'customer_segment'
# (assuming standard forms are 'Enterprise', 'Mid-Market', 'Small Business')
text_standardization_issues = con.execute("""
SELECT DISTINCT customer_segment
FROM customers
WHERE customer_segment NOT IN ('Enterprise', 'Mid-Market', 'Small Business');
""").fetchdf()
print(text_standardization_issues)


print("\nDuplicate 'customer_id' records in customers table:")
# 3. Identify duplicate customer records
duplicate_customers = con.execute("""
SELECT customer_id, COUNT(*) as count
FROM customers
GROUP BY customer_id
HAVING COUNT() > 1;
""").fetchdf()
print(duplicate_customers)


print("\nOutlier invoice amount in invoices table:")
# 4. Find the unusually large invoice (outlier)
outlier_invoice = con.execute("SELECT invoice_id, customer_id, invoice_amount_CUR FROM invoices ORDER BY invoice_amount_CUR DESC LIMIT 1;").fetchdf()
print(outlier_invoice)

In [ ]:
con.close()